# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [32]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [33]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['JDUXTEQAKH', 'VOHNOSDGQI'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[10,  4, 21, 24, 20,  5, 17,  1, 11,  8],
       [22, 15,  8, 14, 15, 19,  4,  7, 17,  9]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  8, 11,  1, 17,  5, 20, 24, 21,  4],
       [ 0,  9, 17,  7,  4, 19, 15, 14,  8, 15]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 8, 11,  1, 17,  5, 20, 24, 21,  4, 10],
       [ 9, 17,  7,  4, 19, 15, 14,  8, 15, 22]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [34]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)

    def build(self, input_shape):
        # Keras 3 对多参数 call 往往只把第一个输入的形状传给 build，形如 (batch, seq)
        if isinstance(input_shape, list) and len(input_shape) == 2 and isinstance(
            input_shape[0], (tuple, list)
        ):
            enc_shape, dec_shape = tuple(input_shape[0]), tuple(input_shape[1])
        else:
            enc_shape = dec_shape = tuple(input_shape)
        self.embed_layer.build(enc_shape)
        enc_seq, dec_seq = enc_shape[1], dec_shape[1]
        self.encoder.build((None, enc_seq, 64))
        self.dense_attn.build((None, enc_seq, self.hidden))
        self.decoder.build((None, dec_seq, 64))
        self.dense.build((None, self.hidden))
        super().build(input_shape)

    def call(self, enc_ids, dec_ids, training=None):
        '''双线性 attention；用定长循环以便 Keras 3 + GradientTape 能正确反传。'''
        enc_emb = self.embed_layer(enc_ids)
        dec_emb = self.embed_layer(dec_ids)
        enc_out, enc_state = self.encoder(enc_emb)
        enc_proj = self.dense_attn(enc_out)
        time_steps = int(dec_ids.shape[1])
        logits_list = []
        state_h = enc_state
        for t in range(time_steps):
            x_t = dec_emb[:, t, :]
            h, new_state = self.decoder_cell(x_t, [state_h])
            state_h = new_state[0]
            scores = tf.reduce_sum(enc_proj * tf.expand_dims(h, axis=1), axis=-1)
            attn_w = tf.nn.softmax(scores, axis=-1)
            context = tf.reduce_sum(tf.expand_dims(attn_w, -1) * enc_out, axis=1)
            logits_list.append(self.dense(h + context))
        logits = tf.stack(logits_list, axis=1)
        return logits

    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,]
        与 call 中单步解码一致：RNN 一步 + 双线性 attention + 输出 logits。
        '''
        inp_emb = self.embed_layer(x)
        if not isinstance(state, (list, tuple)):
            state = [state]
        h, state = self.decoder_cell(inp_emb, state)
        enc_proj = self.dense_attn(enc_out)
        scores = tf.reduce_sum(enc_proj * tf.expand_dims(h, axis=1), axis=-1)
        attn_w = tf.nn.softmax(scores, axis=-1)
        context = tf.reduce_sum(tf.expand_dims(attn_w, -1) * enc_out, axis=1)
        logits = self.dense(h + context)
        out = tf.argmax(logits, axis=-1)
        return out, state

# Loss函数以及训练逻辑

In [35]:
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses


def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [36]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.303225
step 500 : loss 1.400074
step 1000 : loss 0.48932838
step 1500 : loss 0.16579476


<tf.Tensor: shape=(), dtype=float32, numpy=0.07385386526584625>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [37]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
pairs = list(zip(*sequence_reversal()))
print([is_reverse(*item) for item in pairs])
print(pairs)

[True, True, True, True, True, True, True, True, False, False, False, False, True, True, True, True, True, True, False, True, True, True, True, True, True, False, True, False, True, True, False, True]
[('AAQODYBLQDLHDPZUDMOT', 'TOMDUZPDHLDQLBYDOQAA'), ('UVPYQPVCFIVPSWHSNMMQ', 'QMMNSHWSPVIFCVPQYPVU'), ('RIGEALSIBVVCXDGKWUKQ', 'QKUWKGDXCVVBISLAEGIR'), ('WQSTBDVYQNDBMHJGTWQJ', 'JQWTGJHMBDNQYVDBTSQW'), ('MIJIMCPHAKJBQJVYMELQ', 'QLEMYVJQBJKAHPCMIJIM'), ('JBMAILSUWYSIIUVRWDBY', 'YBDWRVUIISYWUSLIAMBJ'), ('PQVMQUNATKVMVCQJOHFB', 'BFHOJQCVMVKTANUQMVQP'), ('GYZQANJOFOGAXVLTQZLM', 'MLZQTLVXAGOFOJNAQZYG'), ('CSNPWASFCEPWVPRSNXWA', 'AWXNSRPVWPECFSAWPGSZ'), ('NIAZVGZZQTIFPIMKIFSK', 'ISFIKMIPFITQZZJVCAVN'), ('GOPBUCLFKGCDQEUKCLUU', 'PULCKUEQDCGKFLCUBPOG'), ('DFVSPNEKGGDYTIKCAUOR', 'ROUACKITYDGGKENPSVFV'), ('HFATJJEKMEYEMDSCZMNL', 'LNMZCSDMEYEMKEJJTAFH'), ('VGWTSQOUXZLOMZHIETJE', 'EJTEIHZMOLZXUOQSTWGV'), ('FHWEWHFLPRMATOJDGBPD', 'DPBGDJOTAMRPLFHWEWHF'), ('RJJWFEUBSJRIQALDJBJX', 'XJBJDLAQIRJSBUEFWJJR')